# Task 044: pyDHM-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Digital holographic microscopy phase reconstruction using angular spectrum method

**Paper**: pyDHM: A Python library for applications in digital holographic microscopy
**Repository**: https://github.com/catrujilla/pyDHM

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 63.87 dB
- **SSIM**: 0.9998


### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import cv2
import os
from math import pi, sqrt, log10

# =============================================================================
# Helper Functions (Defined before use)
# =============================================================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`angularSpectrum`, `PS4`, `quantize_image`, `load_and_preprocess_data`

In [ ]:
def angularSpectrum(field, z, wavelength, dx, dy):
    """
    Function to diffract a complex field using the angular spectrum approximation
    Extracted logic from input code.
    """
    field = np.array(field)
    M, N = field.shape
    x = np.arange(0, N, 1)
    y = np.arange(0, M, 1)
    X, Y = np.meshgrid(x - (N / 2), y - (M / 2), indexing='xy')

    dfx = 1 / (dx * N)
    dfy = 1 / (dy * M)

    field_spec = np.fft.fftshift(field)
    field_spec = np.fft.fft2(field_spec)
    field_spec = np.fft.fftshift(field_spec)

    # Transfer function
    # Note: Using np.exp for phase
    phase_term = np.exp(1j * z * 2 * pi * np.sqrt(np.power(1 / wavelength, 2) - (np.power(X * dfx, 2) + np.power(Y * dfy, 2)) + 0j))

    tmp = field_spec * phase_term

    out = np.fft.ifftshift(tmp)
    out = np.fft.ifft2(out)
    out = np.fft.ifftshift(out)

    return out

def PS4(Inp0, Inp1, Inp2, Inp3):
    '''
    Function to recover the phase information of a sample from four DHM in-axis acquisitions holograms
    '''
    inp0 = np.array(Inp0)
    inp1 = np.array(Inp1)
    inp2 = np.array(Inp2)
    inp3 = np.array(Inp3)

    # compensation process
    # U_obj ~ (I3-I1)j + (I2-I0)
    comp_phase = (inp3 - inp1) * 1j + (inp2 - inp0)

    return comp_phase

def quantize_image(img):
    """Simulates 8-bit quantization."""
    # Assuming the img is the intensity hologram, normalize per set usually, 
    # but here applied locally for simulation logic
    # In the main flow, we will handle max_val properly if available, 
    # or just assume the image range.
    # To keep this function pure, we assume 'img' is the raw intensity.
    # This specific implementation follows the input code logic which normalized 
    # by the max of the set. Since we are inside a helper, we'll do simple 
    # scaling if needed, but the original logic requires context of 4 images.
    # We will implement the quantization logic inside the forward_operator 
    # where all 4 images are available.
    pass 

def load_and_preprocess_data(image_path, target_shape=None):
    """
    Loads the ground truth amplitude, synthesizes phase, and creates the complex field.
    
    Returns:
        field_input (complex np.array): The complex object field.
        gt_amp (np.array): Ground truth amplitude [0, 1].
        gt_phase (np.array): Ground truth phase.
    """
    print(f"Loading Ground Truth Image: {image_path}")
    
    if not os.path.exists(image_path):
        # Trying a few fallbacks as per original logic if exact path fails, 
        # though the caller should ideally provide a valid path.
        # Here we just raise error if it really doesn't exist to be strict.
        raise FileNotFoundError(f"Error: File not found at {image_path}")

    gt_amp = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if gt_amp is None:
        raise ValueError("Error: Failed to read image.")
        
    # Resize if target_shape is provided (optional for flexibility)
    if target_shape is not None:
        gt_amp = cv2.resize(gt_amp, (target_shape[1], target_shape[0]))

    # Normalize to [0, 1]
    gt_amp = gt_amp.astype(np.float32) / 255.0
    
    # Generate Synthetic Phase (e.g., spherical phase)
    M, N = gt_amp.shape
    x = np.arange(0, N, 1)
    y = np.arange(0, M, 1)
    X, Y = np.meshgrid(x - N/2, y - M/2)
    # A simple phase lens-like structure
    gt_phase = 5 * np.exp(-(X**2 + Y**2) / (2 * (200**2))) 
    
    # Create Complex Object
    field_input = gt_amp * np.exp(1j * gt_phase)
    
    return field_input, gt_amp, gt_phase

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(field_input, z, wavelength, dx, dy, add_quantization=True):
    """
    Simulates the hologram recording process:
    1. Propagation to hologram plane.
    2. Interference with phase-shifted reference waves.
    3. (Optional) Camera quantization.
    
    Returns:
        I_stack (list of np.array): List containing [I0, I1, I2, I3].
    """
    # Propagate the field to the hologram plane
    hologram_field_complex = angularSpectrum(field_input, z, wavelength, dx, dy)
    
    # Simulate Reference Wave (Plane Wave, Amplitude 1)
    R = 1.0 
    
    # Phase shifts: 0, pi/2, pi, 3pi/2
    I0 = np.abs(hologram_field_complex + R * np.exp(1j * 0))**2
    I1 = np.abs(hologram_field_complex + R * np.exp(1j * pi/2))**2
    I2 = np.abs(hologram_field_complex + R * np.exp(1j * pi))**2
    I3 = np.abs(hologram_field_complex + R * np.exp(1j * 3*pi/2))**2
    
    if add_quantization:
        # Simulate 8-bit Camera Quantization
        # Normalize by the global maximum of the stack to preserve relative intensities
        max_val = np.max([I0.max(), I1.max(), I2.max(), I3.max()])
        
        def quantize_single(img, limit):
            img_norm = img / limit
            img_8bit = np.round(img_norm * 255).astype(np.uint8)
            # Convert back to float scale
            return img_8bit.astype(np.float32) / 255.0 * limit

        I0 = quantize_single(I0, max_val)
        I1 = quantize_single(I1, max_val)
        I2 = quantize_single(I2, max_val)
        I3 = quantize_single(I3, max_val)
        
    return [I0, I1, I2, I3]

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `run_inversion`

In [ ]:
def run_inversion(I_stack, z, wavelength, dx, dy):
    """
    Performs the reconstruction:
    1. Phase shifting retrieval (PS4).
    2. Back-propagation to object plane.
    
    Returns:
        reconstructed_field (np.complex): The complex object field at z=0.
    """
    I0, I1, I2, I3 = I_stack
    
    # Step 1: Recover Complex Field at Hologram Plane using PS4
    # The PS4 formula returns (I3-I1)j + (I2-I0) which is proportional to 4*R*O
    # Since R=1, we get 4*O. We need to divide by 4 to get true scale.
    recovered_holo_field = PS4(I0, I1, I2, I3) / 4.0
    
    # Step 2: Propagate back to the object plane (-z)
    reconstructed_field = angularSpectrum(recovered_holo_field, -z, wavelength, dx, dy)
    
    return reconstructed_field

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `calculate_psnr`, `calculate_ssim`, `evaluate_results`

In [ ]:
def calculate_psnr(img1, img2):
    """Calculates PSNR between two normalized images."""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    PIXEL_MAX = 1.0 
    return 20 * log10(PIXEL_MAX / sqrt(mse))

def calculate_ssim(img1, img2):
    """
    Calculates SSIM between two images.
    """
    C1 = (0.01 * 1)**2
    C2 = (0.03 * 1)**2
    
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    
    # Gaussian kernel for local mean
    kernel = cv2.getGaussianKernel(11, 1.5)
    window = np.outer(kernel, kernel.transpose())
    
    mu1 = cv2.filter2D(img1, -1, window)[5:-5, 5:-5]
    mu2 = cv2.filter2D(img2, -1, window)[5:-5, 5:-5]
    
    mu1_sq = mu1**2
    mu2_sq = mu2**2
    mu1_mu2 = mu1 * mu2
    
    sigma1_sq = cv2.filter2D(img1**2, -1, window)[5:-5, 5:-5] - mu1_sq
    sigma2_sq = cv2.filter2D(img2**2, -1, window)[5:-5, 5:-5] - mu2_sq
    sigma12 = cv2.filter2D(img1 * img2, -1, window)[5:-5, 5:-5] - mu1_mu2
    
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean()

def evaluate_results(reconstructed_field, gt_amp, gt_phase):
    """
    Calculates metrics and saves images.
    
    Returns:
        metrics (dict): Dictionary containing PSNR and SSIM.
    """
    reconstructed_amplitude = np.abs(reconstructed_field)
    reconstructed_phase = np.angle(reconstructed_field)
    
    # Clip result to valid range [0, 1] for amplitude comparison
    reconstructed_amplitude_clipped = np.clip(reconstructed_amplitude, 0, 1)
    
    psnr_val = calculate_psnr(gt_amp, reconstructed_amplitude_clipped)
    ssim_val = calculate_ssim(gt_amp, reconstructed_amplitude_clipped)
    
    print(f"Amplitude PSNR: {psnr_val:.2f} dB")
    print(f"Amplitude SSIM: {ssim_val:.4f}")
    
    # Save outputs
    cv2.imwrite('output_gt_amp.png', (gt_amp * 255).astype(np.uint8))
    cv2.imwrite('output_reconstruction_amp.png', (reconstructed_amplitude_clipped * 255).astype(np.uint8))
    
    # Visualize Phase (Normalize to 0-255 for visualization)
    gt_phase_norm = ((gt_phase - gt_phase.min()) / (gt_phase.max() - gt_phase.min()) * 255).astype(np.uint8)
    rec_phase_norm = ((reconstructed_phase - reconstructed_phase.min()) / (reconstructed_phase.max() - reconstructed_phase.min()) * 255).astype(np.uint8)
    
    cv2.imwrite('output_gt_phase.png', gt_phase_norm)
    cv2.imwrite('output_reconstruction_phase.png', rec_phase_norm)
    
    print("Saved output images: output_gt_amp.png, output_reconstruction_amp.png, output_gt_phase.png, output_reconstruction_phase.png")
    
    return {"PSNR": psnr_val, "SSIM": ssim_val}

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration ---

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# --- Configuration ---
wavelength = 633e-9  # 633 nm
dx = 5e-6            # 5 um pixel pitch
dy = 5e-6
z = 0.05             # 5 cm propagation distance

# Define Image Path logic (local to script execution)
image_rel_path = 'data/numericalPropagation samples/horse.bmp'
possible_paths = [
    image_rel_path,
    os.path.join(os.getcwd(), 'pyDHM-master', image_rel_path),
    '/data/yjh/pyDHM-master/data/numericalPropagation samples/horse.bmp'
]

image_path = None
for p in possible_paths:
    if os.path.exists(p):
        image_path = p
        break
        
if image_path is None:
    # Generate a dummy image if file not found to strictly allow code to run
    # This prevents crash in environments without the specific file
    print("Warning: 'horse.bmp' not found. Generating synthetic rectangle.")
    dummy_img = np.zeros((512, 512), dtype=np.uint8)
    cv2.rectangle(dummy_img, (150, 150), (350, 350), 255, -1)
    cv2.imwrite('temp_dummy_horse.bmp', dummy_img)
    image_path = 'temp_dummy_horse.bmp'

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
field_input, gt_amp, gt_phase = load_and_preprocess_data(image_path)

### 2. Forward Operator

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward Operator
I_stack = forward_operator(field_input, z, wavelength, dx, dy, add_quantization=True)

# Save one hologram for visualization check
I0_vis = I_stack[0]
cv2.imwrite('output_simulated_hologram_I0.png', (255 * I0_vis / np.max(I0_vis)).astype(np.uint8))

### 3. Run Inversion

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Run Inversion
reconstructed_field = run_inversion(I_stack, z, wavelength, dx, dy)

### 4. Evaluate Results

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate Results
evaluate_results(reconstructed_field, gt_amp, gt_phase)

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pyDHM-master**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=63.87 dB, SSIM=0.9998)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: pyDHM: A Python library for applications in digital holographic microscopy
- Repository: https://github.com/catrujilla/pyDHM
